In [ ]:
from pathlib import Path
import runpy

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pixi.toml").exists())
globals().update(runpy.run_path(str(ROOT / "src" / "repo_paths.py")))

import re
import hashlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from trompy import save_figure_atomic

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 10
mpl.rcParams["savefig.transparent"] = True

In [ ]:
DATAFOLDER = DATA_DIR
df = pd.read_csv(DATAFOLDER / "pellet_retrieval_latency.csv")

In [ ]:
df

In [ ]:
# get proportion of latencies under x s

def get_prop_latencies_under_x(df, x=1):
    
    return (df
            .query("PelletIndex <= 6")
            .assign(under_x=lambda d: d["RetrievalLatency_s"] < x)
            )

x = 5
get_prop_latencies_under_x(df, x=x).under_x.sum() / get_prop_latencies_under_x(df, x=x).shape[0]

In [ ]:
def get_first_n_pellet_latencies(df, n=6, session=None, dietphase=None, pellettype=None):
    
    return (df
            .query("Schedule == @session and PelletIndex <= @n and DietPhase == @dietphase and PelletType == @pellettype")
            .groupby(["MouseID"])
            .RetrievalLatency_s
            .median()
            # .apply(lambda x: x.sort_values("PelletIndex").head(n))
            #.reset_index(drop=True)
           )

# think about filtering out animals with fewer than n pellets

df_fr1 = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="NR1", pellettype="Grain")
df_progratio = get_first_n_pellet_latencies(df, n=6, session="ProgRatio", dietphase="NR1", pellettype="Grain")

In [ ]:
df_fr1_nr1_grain = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="NR1", pellettype="Grain")
df_fr1_nr1_sucrose = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="NR1", pellettype="Sucrose")
df_fr1_pr_grain = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="PR", pellettype="Grain")
df_fr1_pr_sucrose = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="PR", pellettype="Sucrose")
df_fr1_nr2_grain = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="NR2", pellettype="Grain")
df_fr1_nr2_sucrose = get_first_n_pellet_latencies(df, n=6, session="FR1", dietphase="NR2", pellettype="Sucrose")

In [ ]:
from scipy.stats import ttest_ind

# NR1
print(f"NR1: {ttest_ind(df_fr1_nr1_grain, df_fr1_nr1_sucrose)}")
          
# PR
print(f"PR: {ttest_ind(df_fr1_pr_grain, df_fr1_pr_sucrose)}")

# NR2
print(f"NR2: {ttest_ind(df_fr1_nr2_grain, df_fr1_nr2_sucrose)}")

In [ ]:
(df.query("Schedule == 'FR1' and PelletIndex <= 6")
 .groupby(["MouseID", "DietPhase", "PelletType"], as_index=False)["RetrievalLatency_s"]
 .median()
 .groupby(["DietPhase", "PelletType"])["RetrievalLatency_s"]
 .agg(["mean", "sem"]))